# Experiment 1 — iTransformer Hard-Mask Transfer on All 20 Matched Conditions

This notebook evaluates the complete matched hard-mask experiment. It expands the original 8-condition diagnostic screen to every condition for which the matched Phase-1 iTransformer checkpoint suite is available: 5 general datasets × 4 prediction lengths = **20 conditions**.

- Same Phase-1 checkpoints and preprocessing.
- Same training-only predictive-utility teacher and dimension-aware source budget.
- Same last-block hard mask and matched fine-tuning protocol.
- DenseFineTune, SharedSparse, and HorizonAdaptiveSparse are evaluated for every condition.

PeMS is not included because the retained Phase-1 strong-backbone checkpoint suite covers the five general benchmarks only. Adding PeMS would require an additional backbone-training study rather than a pure expansion of the matched transfer experiment.


In [1]:
from pathlib import Path
from types import SimpleNamespace

import copy
import gc
import importlib
import math
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 2026

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


PyTorch: 2.4.1+cu121
Device: cuda
GPU: NVIDIA A100-SXM4-80GB


## 1. Fixed screening configuration


In [2]:
SEQ_LEN = 96
LABEL_LEN = 48

ALL_HORIZONS = [
    96,
    192,
    336,
    720,
]

# Matched positive/negative diagnostic screening.
CONDITIONS = [
    (dataset_name, H)
    for dataset_name in [
        "Electricity", "Weather", "Solar", "ETTh1", "ETTm1"
    ]
    for H in ALL_HORIZONS
]

VARIANTS = [
    "DenseFineTune",
    "SharedSparse",
    "HorizonAdaptiveSparse",
]

MAX_TOPK = 10

# Predictive-Utility Teacher fallback.
# Existing legacy caches may have been saved with only 2 or a few sources.
# For this hard-mask experiment we must preserve the fixed K(C) protocol,
# so an incompatible cache is rebuilt from official TRAIN data only.
MAX_TEACHER_TRAIN_ORIGINS = 5000
MAX_TEACHER_SOURCE_CANDIDATES = 128
TEACHER_FIT_FRACTION = 0.70
TEACHER_RIDGE_LAMBDA = 1e-3
TEACHER_SOURCE_RIDGE_LAMBDA = 1e-2

# Fine-tune only last encoder block + forecast projection.
FINETUNE_LR = 1e-4
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 8
PATIENCE = 3
GRAD_CLIP = 1.0

NUM_WORKERS = 2
RESUME = True

PROJECT_ROOT_CANDIDATES = [
    Path("/data/code/2026_08"),
    Path.cwd(),
]

PROJECT_ROOT = next(
    (
        p
        for p in PROJECT_ROOT_CANDIDATES
        if p.exists()
    ),
    Path.cwd(),
).resolve()

BASELINE_ROOT = (
    PROJECT_ROOT
    / "results_external_baselines_phase1_direct"
)

ROLLING_ROOT = (
    PROJECT_ROOT
    / "results_rolling_temporal_stability_gate"
)

RIDGE_REFERENCE = (
    ROLLING_ROOT
    / "stability_vs_controls.csv"
)

ORIGINAL_OUTPUT_DIR = (
    PROJECT_ROOT
    / "results_itransformer_explicit_sparse_horizon_mask_v5"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results_itransformer_explicit_sparse_horizon_mask_all20"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Conditions:", CONDITIONS)
print("Variants:", VARIANTS)


PROJECT_ROOT: /data/code/2026_08
OUTPUT_DIR: /data/code/2026_08/results_itransformer_explicit_sparse_horizon_mask_all20
Conditions: [('Electricity', 96), ('Electricity', 192), ('Electricity', 336), ('Electricity', 720), ('Weather', 96), ('Weather', 192), ('Weather', 336), ('Weather', 720), ('Solar', 96), ('Solar', 192), ('Solar', 336), ('Solar', 720), ('ETTh1', 96), ('ETTh1', 192), ('ETTh1', 336), ('ETTh1', 720), ('ETTm1', 96), ('ETTm1', 192), ('ETTm1', 336), ('ETTm1', 720)]
Variants: ['DenseFineTune', 'SharedSparse', 'HorizonAdaptiveSparse']


## 2. Determinism and batch size


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def choose_batch_size(
    dataset_name,
    H,
):
    batch = 32

    if dataset_name == "Electricity":
        batch = 16

        if H >= 336:
            batch = 8

    elif dataset_name == "Solar":
        batch = 16

    elif dataset_name == "Weather":
        batch = 32

    return batch


def effective_topk(C):
    return int(
        max(
            1,
            min(
                MAX_TOPK,
                math.ceil(
                    (C - 1)
                    /
                    2
                ),
            ),
        )
    )


for C in [
    7,
    21,
    137,
    321,
]:
    print(
        f"C={C:3d} -> K={effective_topk(C)}"
    )


C=  7 -> K=3
C= 21 -> K=10
C=137 -> K=10
C=321 -> K=10


## 3. Locate Time-Series-Library


In [4]:
TSLIB_CANDIDATES = [
    Path(
        "/data/Time-Series-Library"
    ),
    Path(
        "/data/Time-Series-Library_v2"
    ),
    Path(
        "/data/Time-Series-Library"
    ),
    PROJECT_ROOT
    / "Time-Series-Library",
    PROJECT_ROOT
    / "Time-Series-Library_v2",
]


def is_tslib_root(p):
    p = Path(p)

    return (
        p.is_dir()
        and
        (
            p
            / "models"
            / "iTransformer.py"
        ).exists()
        and
        (
            p
            / "layers"
            / "Transformer_EncDec.py"
        ).exists()
        and
        (
            p
            / "layers"
            / "SelfAttention_Family.py"
        ).exists()
    )


TSL_ROOT = next(
    (
        p.resolve()
        for p in TSLIB_CANDIDATES
        if is_tslib_root(p)
    ),
    None,
)

if TSL_ROOT is None:
    raise FileNotFoundError(
        "Time-Series-Library source not found."
    )

if str(TSL_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(TSL_ROOT),
    )

iTransformer_module = (
    importlib.import_module(
        "models.iTransformer"
    )
)

print("TSLib:", TSL_ROOT)


TSLib: /data/Time-Series-Library


## 4. Dataset protocol — exactly aligned with Phase-1


In [5]:
DATASET_CANDIDATES = {
    "Electricity": [
        Path(
            "/data/dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/dataset/ECL/electricity.csv"
        ),
        Path(
            "/data/dataset/electricity.csv"
        ),
    ],

    "Weather": [
        Path(
            "/data/dataset/weather/weather.csv"
        ),
        Path(
            "/data/dataset/weather.csv"
        ),
    ],

    "Solar": [
        Path(
            "/data/dataset/solar/solar_AL.txt"
        ),
        Path(
            "/data/dataset/Solar/solar_AL.txt"
        ),
        Path(
            "/data/dataset/solar_AL.txt"
        ),
    ],

    "ETTh1": [
        Path(
            "/data/dataset/ETT-small/ETTh1.csv"
        ),
        Path(
            "/data/dataset/ETTh1.csv"
        ),
    ],

    "ETTm1": [
        Path(
            "/data/dataset/ETT-small/ETTm1.csv"
        ),
        Path(
            "/data/dataset/ETTm1.csv"
        ),
    ],
}


DATASET_FILES = {}

for dataset_name, candidates in (
    DATASET_CANDIDATES.items()
):
    found = next(
        (
            p.resolve()
            for p in candidates
            if p.exists()
        ),
        None,
    )

    if found is None:
        raise FileNotFoundError(
            f"{dataset_name}: "
            f"dataset file not found."
        )

    DATASET_FILES[
        dataset_name
    ] = found


DATASET_SPECS = {
    "Electricity": {
        "channels": 321,
        "freq": "h",
    },

    "Weather": {
        "channels": 21,
        "freq": "t",
    },

    "Solar": {
        "channels": 137,
        "freq": "h",
    },

    "ETTh1": {
        "channels": 7,
        "freq": "h",
    },

    "ETTm1": {
        "channels": 7,
        "freq": "t",
    },
}


for k, v in DATASET_FILES.items():
    print(
        f"{k:12s}",
        "->",
        v
    )


Electricity  -> /data/dataset/electricity/electricity.csv
Weather      -> /data/dataset/weather/weather.csv
Solar        -> /data/dataset/solar/solar_AL.txt
ETTh1        -> /data/dataset/ETT-small/ETTh1.csv
ETTm1        -> /data/dataset/ETT-small/ETTm1.csv


In [6]:
def load_numeric_series(
    dataset_name,
    path,
):
    path = Path(path)

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)

        date_col = None

        for candidate in [
            "date",
            "datetime",
            "timestamp",
            "time",
        ]:
            if candidate in df.columns:
                date_col = candidate
                break

        if date_col is None:
            first_col = df.columns[0]

            if not pd.api.types.is_numeric_dtype(
                df[first_col]
            ):
                date_col = first_col

        if date_col is not None:
            df = df.drop(
                columns=[date_col]
            )

        df = df.select_dtypes(
            include=[np.number]
        )

        x = df.to_numpy(
            dtype=np.float32
        )

    else:
        try:
            x = np.loadtxt(
                path,
                delimiter=",",
                dtype=np.float32,
            )

        except Exception:
            x = np.loadtxt(
                path,
                dtype=np.float32,
            )

        if x.ndim == 1:
            x = x[:, None]

    if x.ndim != 2:
        raise ValueError(
            f"{dataset_name}: unexpected shape {x.shape}"
        )

    if not np.isfinite(x).all():
        raise ValueError(
            f"{dataset_name}: non-finite values"
        )

    return x


def split_boundaries(
    dataset_name,
    T,
):
    if dataset_name == "ETTh1":
        train_end = (
            12
            *
            30
            *
            24
        )

        val_end = (
            train_end
            +
            4
            *
            30
            *
            24
        )

        test_end = (
            val_end
            +
            4
            *
            30
            *
            24
        )

        return (
            min(train_end, T),
            min(val_end, T),
            min(test_end, T),
        )

    if dataset_name == "ETTm1":
        unit = (
            30
            *
            24
            *
            4
        )

        train_end = (
            12
            *
            unit
        )

        val_end = (
            train_end
            +
            4
            *
            unit
        )

        test_end = (
            val_end
            +
            4
            *
            unit
        )

        return (
            min(train_end, T),
            min(val_end, T),
            min(test_end, T),
        )

    return (
        int(
            0.70
            *
            T
        ),
        int(
            0.80
            *
            T
        ),
        T,
    )


def load_and_normalize(
    dataset_name,
):
    raw = load_numeric_series(
        dataset_name,
        DATASET_FILES[
            dataset_name
        ],
    )

    T, C = raw.shape

    (
        train_end,
        val_end,
        test_end,
    ) = split_boundaries(
        dataset_name,
        T,
    )

    mean = raw[
        :train_end
    ].mean(
        axis=0,
        keepdims=True,
    )

    std = raw[
        :train_end
    ].std(
        axis=0,
        keepdims=True,
    )

    std = np.maximum(
        std,
        1e-6,
    )

    x = (
        raw
        -
        mean
    ) / std

    return (
        x.astype(
            np.float32
        ),
        {
            "T": T,
            "C": C,
            "train_end": train_end,
            "val_end": val_end,
            "test_end": test_end,
        },
    )


class ForecastDataset(Dataset):
    def __init__(
        self,
        data,
        seq_len,
        label_len,
        pred_len,
        target_start_min,
        target_end_exclusive,
    ):
        self.data = torch.from_numpy(
            data
        ).float()

        self.seq_len = int(
            seq_len
        )

        self.label_len = int(
            label_len
        )

        self.pred_len = int(
            pred_len
        )

        first_t = max(
            self.seq_len,
            self.label_len,
            int(
                target_start_min
            ),
        )

        last_t = (
            int(
                target_end_exclusive
            )
            -
            self.pred_len
        )

        self.origins = np.arange(
            first_t,
            last_t + 1,
            dtype=np.int64,
        )

        if len(
            self.origins
        ) == 0:
            raise RuntimeError(
                "No valid forecasting windows."
            )

    def __len__(self):
        return len(
            self.origins
        )

    def __getitem__(
        self,
        idx,
    ):
        t = int(
            self.origins[
                idx
            ]
        )

        x = self.data[
            t-self.seq_len:
            t
        ]

        y = self.data[
            t-self.label_len:
            t+self.pred_len
        ]

        return x, y


def prepare_datasets(
    dataset_name,
    H,
):
    x, meta = (
        load_and_normalize(
            dataset_name
        )
    )

    expected_C = (
        DATASET_SPECS[
            dataset_name
        ][
            "channels"
        ]
    )

    if meta["C"] != expected_C:
        raise RuntimeError(
            f"{dataset_name}: "
            f"detected C={meta['C']}, "
            f"expected C={expected_C}"
        )

    train_ds = ForecastDataset(
        x,
        SEQ_LEN,
        LABEL_LEN,
        H,
        target_start_min=SEQ_LEN,
        target_end_exclusive=meta[
            "train_end"
        ],
    )

    val_ds = ForecastDataset(
        x,
        SEQ_LEN,
        LABEL_LEN,
        H,
        target_start_min=meta[
            "train_end"
        ],
        target_end_exclusive=meta[
            "val_end"
        ],
    )

    test_ds = ForecastDataset(
        x,
        SEQ_LEN,
        LABEL_LEN,
        H,
        target_start_min=meta[
            "val_end"
        ],
        target_end_exclusive=meta[
            "test_end"
        ],
    )

    return (
        x,
        train_ds,
        val_ds,
        test_ds,
        meta,
    )


def make_loaders(
    train_ds,
    val_ds,
    test_ds,
    batch_size,
    seed,
):
    generator = torch.Generator()

    generator.manual_seed(
        seed
        +
        100000
    )

    common = dict(
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        **common,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        **common,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        **common,
    )

    return (
        train_loader,
        val_loader,
        test_loader,
    )


## 5. Load exact Phase-1 iTransformer


In [7]:
def build_itransformer_args(
    dataset_name,
    H,
):
    spec = DATASET_SPECS[
        dataset_name
    ]

    return SimpleNamespace(
        task_name="long_term_forecast",
        model="iTransformer",

        seq_len=SEQ_LEN,
        label_len=LABEL_LEN,
        pred_len=H,

        enc_in=spec[
            "channels"
        ],
        dec_in=spec[
            "channels"
        ],
        c_out=spec[
            "channels"
        ],

        e_layers=3,
        d_layers=1,
        n_heads=8,
        d_model=512,
        d_ff=512,
        factor=3,
        dropout=0.1,

        embed="timeF",
        freq=spec[
            "freq"
        ],
        activation="gelu",
        output_attention=False,

        class_strategy="projection",
        use_norm=1,

        moving_avg=25,
        distil=True,
        top_k=5,
        num_kernels=6,
        seasonal_patterns="Monthly",
        inverse=False,
    )


def condition_dir(
    dataset_name,
    H,
):
    return (
        BASELINE_ROOT
        / "iTransformer"
        / dataset_name
        / f"H{H}"
    )


def build_and_load_itransformer(
    dataset_name,
    H,
):
    args = build_itransformer_args(
        dataset_name,
        H,
    )

    model = (
        iTransformer_module
        .Model(args)
        .float()
        .to(DEVICE)
    )

    checkpoint = (
        condition_dir(
            dataset_name,
            H,
        )
        / "best_model.pt"
    )

    if not checkpoint.exists():
        raise FileNotFoundError(
            checkpoint
        )

    state = torch.load(
        checkpoint,
        map_location="cpu",
    )

    model.load_state_dict(
        state,
        strict=True,
    )

    model.eval()

    return model


def load_phase1_result(
    dataset_name,
    H,
):
    path = (
        condition_dir(
            dataset_name,
            H,
        )
        / "result.csv"
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    df = pd.read_csv(
        path
    )

    if len(df) != 1:
        raise RuntimeError(
            f"Unexpected Phase-1 rows: {path}"
        )

    return (
        df.iloc[0]
        .to_dict()
    )


## 6. Predictive-Utility Teacher caches

This section implements the matched horizon-specific topology-transfer mechanism. Predictive-utility source sets are computed from training data and then imposed on the neural forecaster as specified in the paper.


In [8]:
TEACHER_SEARCH_ROOTS = [
    PROJECT_ROOT
    / "results_itransformer_lowrank_horizon_routing_screening"
    / "predictive_teacher_cache",

    PROJECT_ROOT
    / "results_itransformer_horizon_routing_screening"
    / "predictive_teacher_cache",

    PROJECT_ROOT
    / "results_final_protocol_hybrid0_H96_H720"
    / "predictive_teacher_cache",

    PROJECT_ROOT
    / "results_intermediate_horizon_hybrid_router"
    / "predictive_teacher_cache",
]

# New cache dedicated to this explicit hard-mask experiment.
# It always stores at least K(C) unique predictive sources per target.
# Reuse the expensive teacher cache independently of v5 result files.
HARDMASK_TEACHER_DIR = (
    PROJECT_ROOT
    / "results_itransformer_explicit_sparse_horizon_mask"
    / "predictive_teacher_hardmask_cache"
)

HARDMASK_TEACHER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def legacy_teacher_filename(
    dataset_name,
    H,
):
    return (
        f"{dataset_name}_H{H}_predictive_topk.npz"
    )


def find_legacy_teacher_cache(
    dataset_name,
    H,
):
    """
    Legacy caches are OPTIONAL.

    They are reused only when they contain enough unique valid source
    channels for the current K(C). Otherwise the new teacher cache is
    rebuilt from official training data.
    """
    filename = legacy_teacher_filename(
        dataset_name,
        H,
    )

    for root in TEACHER_SEARCH_ROOTS:
        path = root / filename

        if path.exists():
            return path

    matches = list(
        PROJECT_ROOT.glob(
            f"results*/**/{filename}"
        )
    )

    if matches:
        return matches[0]

    return None


def hardmask_teacher_path(
    dataset_name,
    H,
    K,
):
    return (
        HARDMASK_TEACHER_DIR
        / (
            f"{dataset_name}_H{H}_"
            f"endpoint_predictive_topk_K{K}.npz"
        )
    )


needed_datasets = sorted(
    {
        d
        for d, _
        in CONDITIONS
    }
)

print(
    "Legacy teacher caches are optional."
)

print(
    "Incompatible caches will be rebuilt automatically under:"
)

print(
    HARDMASK_TEACHER_DIR
)

for d in needed_datasets:
    print(
        "\n",
        d
    )

    for H in ALL_HORIZONS:
        legacy = find_legacy_teacher_cache(
            d,
            H,
        )

        print(
            f" H={H}:",
            (
                str(legacy)
                if legacy is not None
                else "no legacy cache -> rebuild"
            ),
        )


Legacy teacher caches are optional.
Incompatible caches will be rebuilt automatically under:
/data/code/2026_08/results_itransformer_explicit_sparse_horizon_mask/predictive_teacher_hardmask_cache

 ETTh1
 H=96: /data/code/2026_08/results_oracle_predictive_routing/oracle_cache/ETTh1_H96_predictive_topk.npz
 H=192: /data/code/2026_08/results_intermediate_horizon_hybrid_router/predictive_teacher_cache/ETTh1_H192_predictive_topk.npz
 H=336: /data/code/2026_08/results_intermediate_horizon_hybrid_router/predictive_teacher_cache/ETTh1_H336_predictive_topk.npz
 H=720: /data/code/2026_08/results_oracle_predictive_routing/oracle_cache/ETTh1_H720_predictive_topk.npz

 ETTm1
 H=96: /data/code/2026_08/results_oracle_predictive_routing/oracle_cache/ETTm1_H96_predictive_topk.npz
 H=192: /data/code/2026_08/results_intermediate_horizon_hybrid_router/predictive_teacher_cache/ETTm1_H192_predictive_topk.npz
 H=336: /data/code/2026_08/results_intermediate_horizon_hybrid_router/predictive_teacher_cache/ETTm

## 7. Build endpoint source rankings and masks


In [9]:
# ---------------------------------------------------------------------
# Predictive-Utility Teacher fallback
# ---------------------------------------------------------------------
#
# Why this exists:
# Some legacy teacher caches were generated for earlier router experiments
# with a small cached Top-K (e.g., 2). They cannot support the present
# explicit sparse mask with K(C)=10.
#
# We DO NOT reduce K to fit the old cache because that would change the
# scientific protocol. Instead:
#
#   1) reuse a legacy cache only if every target has >= K unique sources;
#   2) otherwise rebuild the teacher using official TRAIN data only;
#   3) save a new endpoint Top-K cache dedicated to this experiment.
#
# ---------------------------------------------------------------------


def teacher_evenly_subsample(
    values,
    max_n,
):
    values = np.asarray(
        values,
        dtype=np.int64,
    )

    if len(values) <= max_n:
        return values

    idx = np.linspace(
        0,
        len(values) - 1,
        max_n,
        dtype=np.int64,
    )

    return values[idx]


def teacher_train_origins(
    train_end,
):
    """
    Use one common origin set valid for ALL_HORIZONS.
    This makes H=96/192/336/720 endpoint teacher rankings directly
    comparable when building the Shared consensus topology.
    """
    max_h = max(
        ALL_HORIZONS
    )

    first = SEQ_LEN
    last = (
        int(train_end)
        -
        int(max_h)
    )

    if last < first:
        raise RuntimeError(
            f"Insufficient train interval for max horizon {max_h}: "
            f"train_end={train_end}"
        )

    origins = np.arange(
        first,
        last + 1,
        dtype=np.int64,
    )

    return teacher_evenly_subsample(
        origins,
        MAX_TEACHER_TRAIN_ORIGINS,
    )


def teacher_summary_features(
    x,
    origins,
):
    """
    Same compact features used by the controlled predictive-utility
    experiments:
      [last, mean3, mean12, delta12]
    """
    origins = np.asarray(
        origins,
        dtype=np.int64,
    )

    last = x[
        origins - 1
    ]

    mean3 = np.stack(
        [
            x[t-3:t].mean(
                axis=0
            )
            for t in origins
        ],
        axis=0,
    )

    mean12 = np.stack(
        [
            x[t-12:t].mean(
                axis=0
            )
            for t in origins
        ],
        axis=0,
    )

    delta12 = (
        x[
            origins - 1
        ]
        -
        x[
            origins - 12
        ]
    )

    return np.stack(
        [
            last,
            mean3,
            mean12,
            delta12,
        ],
        axis=-1,
    ).astype(
        np.float32
    )


def teacher_add_bias(
    X,
):
    return np.concatenate(
        [
            np.ones(
                (
                    len(X),
                    1,
                ),
                dtype=np.float32,
            ),
            X.astype(
                np.float32
            ),
        ],
        axis=1,
    )


def teacher_ridge_fit(
    X,
    y,
    lam=TEACHER_RIDGE_LAMBDA,
):
    Xb = teacher_add_bias(
        X
    ).astype(
        np.float64
    )

    p = Xb.shape[
        1
    ]

    reg = np.eye(
        p,
        dtype=np.float64,
    )

    reg[
        0,
        0
    ] = 0.0

    A = Xb.T @ Xb
    b = (
        Xb.T
        @
        y.astype(
            np.float64
        )
    )

    return np.linalg.solve(
        A
        +
        lam
        *
        reg,
        b,
    )


def teacher_ridge_predict(
    X,
    beta,
):
    return (
        teacher_add_bias(
            X
        )
        @
        beta
    ).astype(
        np.float64
    )


def teacher_candidate_map(
    train_features,
):
    """
    Broad TRAIN-ONLY candidate pool.

    Current-value absolute correlation is used only as a computational
    pre-screen. Predictive utility determines the final ranking.
    """
    current = train_features[
        :,
        :,
        0,
    ].astype(
        np.float64
    )

    current = (
        current
        -
        current.mean(
            axis=0,
            keepdims=True,
        )
    )

    denom = np.sqrt(
        np.sum(
            current
            *
            current,
            axis=0,
        )
    )

    denom = np.maximum(
        denom,
        1e-12,
    )

    normed = (
        current
        /
        denom[
            None,
            :
        ]
    )

    C = current.shape[
        1
    ]

    pool_size = min(
        MAX_TEACHER_SOURCE_CANDIDATES,
        max(
            1,
            C - 1,
        ),
    )

    result = {}

    for i in range(C):
        corr = (
            normed[
                :,
                i
            ]
            @
            normed
        )

        corr = np.abs(
            corr
        )

        corr[
            i
        ] = -np.inf

        idx = np.argpartition(
            corr,
            -pool_size,
        )[
            -pool_size:
        ]

        idx = idx[
            np.argsort(
                corr[
                    idx
                ]
            )[
                ::-1
            ]
        ]

        result[
            i
        ] = idx.astype(
            np.int64
        )

    return result


def teacher_predictive_utility(
    X_all,
    y,
    target_idx,
    source_ids,
):
    """
    Predictive-Utility Teacher.

    Fit/score is chronological and remains entirely inside official TRAIN.

    For each candidate source:
      - fit target-only Ridge;
      - residualize source summary features against target features;
      - fit a marginal residual correction;
      - score percentage MSE reduction on held-out teacher-score data.
    """
    N = X_all.shape[
        0
    ]

    split = int(
        TEACHER_FIT_FRACTION
        *
        N
    )

    split = min(
        max(
            split,
            32,
        ),
        N - 16,
    )

    fit_idx = np.arange(
        0,
        split,
        dtype=np.int64,
    )

    score_idx = np.arange(
        split,
        N,
        dtype=np.int64,
    )

    Xt = X_all[
        :,
        target_idx,
        :
    ].astype(
        np.float64
    )

    U = X_all[
        :,
        source_ids,
        :
    ].astype(
        np.float64
    )

    y = y.astype(
        np.float64
    )

    # Target-only baseline.
    beta_t = teacher_ridge_fit(
        Xt[
            fit_idx
        ],
        y[
            fit_idx
        ],
    )

    pred_fit = teacher_ridge_predict(
        Xt[
            fit_idx
        ],
        beta_t,
    )

    pred_score = teacher_ridge_predict(
        Xt[
            score_idx
        ],
        beta_t,
    )

    r_fit = (
        y[
            fit_idx
        ]
        -
        pred_fit
    )

    r_score = (
        y[
            score_idx
        ]
        -
        pred_score
    )

    base_mse = float(
        np.mean(
            r_score
            *
            r_score
        )
    )

    # Residualize all source features against target history.
    Xt_fit_b = teacher_add_bias(
        Xt[
            fit_idx
        ]
    ).astype(
        np.float64
    )

    Xt_score_b = teacher_add_bias(
        Xt[
            score_idx
        ]
    ).astype(
        np.float64
    )

    U_fit = U[
        fit_idx
    ]

    U_score = U[
        score_idx
    ]

    S = U_fit.shape[
        1
    ]

    Fdim = U_fit.shape[
        2
    ]

    U_fit_flat = U_fit.reshape(
        len(
            fit_idx
        ),
        S
        *
        Fdim,
    )

    A = (
        Xt_fit_b.T
        @
        Xt_fit_b
    )

    reg = np.eye(
        A.shape[
            0
        ],
        dtype=np.float64,
    )

    reg[
        0,
        0
    ] = 0.0

    coef_u = np.linalg.solve(
        A
        +
        TEACHER_RIDGE_LAMBDA
        *
        reg,
        Xt_fit_b.T
        @
        U_fit_flat,
    )

    U_fit_res = (
        U_fit_flat
        -
        Xt_fit_b
        @
        coef_u
    ).reshape(
        len(
            fit_idx
        ),
        S,
        Fdim,
    )

    U_score_flat = U_score.reshape(
        len(
            score_idx
        ),
        S
        *
        Fdim,
    )

    U_score_res = (
        U_score_flat
        -
        Xt_score_b
        @
        coef_u
    ).reshape(
        len(
            score_idx
        ),
        S,
        Fdim,
    )

    gram = np.einsum(
        "nsf,nsg->sfg",
        U_fit_res,
        U_fit_res,
    )

    cross = np.einsum(
        "nsf,n->sf",
        U_fit_res,
        r_fit,
    )

    eye = np.eye(
        Fdim,
        dtype=np.float64,
    )[
        None,
        :,
        :
    ]

    beta_s = np.linalg.solve(
        gram
        +
        TEACHER_SOURCE_RIDGE_LAMBDA
        *
        eye,
        cross[
            :,
            :,
            None
        ],
    )[
        :,
        :,
        0
    ]

    pred_res = np.einsum(
        "nsf,sf->ns",
        U_score_res,
        beta_s,
    )

    err = (
        r_score[
            :,
            None
        ]
        -
        pred_res
    )

    mse_source = np.mean(
        err
        *
        err,
        axis=0,
    )

    utility = (
        100.0
        *
        (
            base_mse
            -
            mse_source
        )
        /
        max(
            base_mse,
            1e-12,
        )
    )

    return utility.astype(
        np.float32
    )


def _top_unique_valid_from_arrays(
    source_idx,
    source_gain,
    target_idx,
    C,
    K,
):
    source_idx = np.asarray(
        source_idx,
        dtype=np.int64,
    ).reshape(
        -1
    )

    source_gain = np.asarray(
        source_gain,
        dtype=np.float64,
    ).reshape(
        -1
    )

    valid = (
        (source_idx >= 0)
        &
        (source_idx < C)
        &
        (source_idx != target_idx)
        &
        np.isfinite(
            source_gain
        )
    )

    ids = source_idx[
        valid
    ]

    gains = source_gain[
        valid
    ]

    if len(
        ids
    ) == 0:
        return (
            np.array(
                [],
                dtype=np.int64,
            ),
            np.array(
                [],
                dtype=np.float32,
            ),
        )

    order = np.argsort(
        gains
    )[
        ::-1
    ]

    seen = set()
    out_ids = []
    out_gains = []

    for q in order:
        j = int(
            ids[
                q
            ]
        )

        if j in seen:
            continue

        seen.add(
            j
        )

        out_ids.append(
            j
        )

        out_gains.append(
            float(
                gains[
                    q
                ]
            )
        )

        if len(
            out_ids
        ) >= K:
            break

    return (
        np.asarray(
            out_ids,
            dtype=np.int64,
        ),
        np.asarray(
            out_gains,
            dtype=np.float32,
        ),
    )


def _extract_legacy_endpoint(
    legacy_path,
    H,
    C,
    K,
):
    """
    Return compatible [C,K] source_idx/source_gain or None.
    """
    if legacy_path is None:
        return None

    try:
        cache = np.load(
            legacy_path
        )

        if (
            "oracle_idx"
            not in cache
            or
            "oracle_gain"
            not in cache
        ):
            return None

        idx = cache[
            "oracle_idx"
        ]

        gain = cache[
            "oracle_gain"
        ]

        # Legacy cache is usually [H,C,K_cache].
        if idx.ndim == 3:
            if (
                idx.shape[
                    0
                ]
                <
                H
                or
                idx.shape[
                    1
                ]
                !=
                C
            ):
                return None

            idx = idx[
                H - 1
            ]

            gain = gain[
                H - 1
            ]

        elif idx.ndim == 2:
            if idx.shape[
                0
            ] != C:
                return None

        else:
            return None

        out_idx = np.zeros(
            (
                C,
                K,
            ),
            dtype=np.int64,
        )

        out_gain = np.zeros(
            (
                C,
                K,
            ),
            dtype=np.float32,
        )

        for i in range(C):
            ids, gains = (
                _top_unique_valid_from_arrays(
                    idx[
                        i
                    ],
                    gain[
                        i
                    ],
                    i,
                    C,
                    K,
                )
            )

            if len(
                ids
            ) < K:
                # This is the exact failure in v1:
                # legacy cache does not contain enough unique sources.
                return None

            out_idx[
                i
            ] = ids

            out_gain[
                i
            ] = gains

        return (
            out_idx,
            out_gain,
        )

    except Exception:
        return None


def rebuild_dataset_hardmask_teacher(
    dataset_name,
):
    """
    Build endpoint Top-K predictive source rankings for ALL_HORIZONS.

    IMPORTANT:
      - official TRAIN data only;
      - same K(C) as the hard-mask experiment;
      - one shared origin set valid through H=720;
      - results cached on disk for resume.
    """
    print(
        "\n"
        +
        "="
        *
        88
    )

    print(
        f"Rebuilding hard-mask Predictive-Utility Teacher: "
        f"{dataset_name}"
    )

    print(
        "="
        *
        88
    )

    x, meta = load_and_normalize(
        dataset_name
    )

    C = int(
        meta[
            "C"
        ]
    )

    K = effective_topk(
        C
    )

    origins = teacher_train_origins(
        meta[
            "train_end"
        ]
    )

    X = teacher_summary_features(
        x,
        origins,
    )

    candidate_map = teacher_candidate_map(
        X
    )

    print(
        f"C={C}, K={K}, origins={len(origins)}, "
        f"candidate_pool<={MAX_TEACHER_SOURCE_CANDIDATES}"
    )

    # Allocate all horizons once.
    top_idx = {
        H: np.zeros(
            (
                C,
                K,
            ),
            dtype=np.int64,
        )
        for H in ALL_HORIZONS
    }

    top_gain = {
        H: np.zeros(
            (
                C,
                K,
            ),
            dtype=np.float32,
        )
        for H in ALL_HORIZONS
    }

    for i in range(C):
        source_ids = candidate_map[
            i
        ]

        for H in ALL_HORIZONS:
            y = x[
                origins
                +
                H
                -
                1,
                i,
            ]

            utility = teacher_predictive_utility(
                X,
                y,
                i,
                source_ids,
            )

            # Sort by predictive utility.
            order = np.argsort(
                utility
            )[
                ::-1
            ]

            selected_local = order[
                :K
            ]

            selected_ids = source_ids[
                selected_local
            ]

            selected_gain = utility[
                selected_local
            ]

            if len(
                np.unique(
                    selected_ids
                )
            ) != K:
                raise RuntimeError(
                    f"{dataset_name} H={H} target={i}: "
                    f"teacher rebuild produced duplicate sources."
                )

            top_idx[
                H
            ][
                i
            ] = selected_ids

            top_gain[
                H
            ][
                i
            ] = selected_gain

        if (
            (i + 1)
            %
            max(
                1,
                C // 8,
            )
            ==
            0
            or
            i + 1
            ==
            C
        ):
            print(
                f"  targets {i+1}/{C}"
            )

    for H in ALL_HORIZONS:
        path = hardmask_teacher_path(
            dataset_name,
            H,
            K,
        )

        np.savez_compressed(
            path,
            source_idx=top_idx[
                H
            ],
            source_gain=top_gain[
                H
            ],
            dataset=np.asarray(
                dataset_name
            ),
            pred_len=np.asarray(
                H,
                dtype=np.int64,
            ),
            channels=np.asarray(
                C,
                dtype=np.int64,
            ),
            topk=np.asarray(
                K,
                dtype=np.int64,
            ),
            train_origin_count=np.asarray(
                len(
                    origins
                ),
                dtype=np.int64,
            ),
        )

        print(
            "  saved:",
            path
        )

    del (
        x,
        X,
        candidate_map,
        top_idx,
        top_gain,
    )

    gc.collect()


def ensure_hardmask_teacher_cache(
    dataset_name,
    H,
    C,
    K,
):
    """
    Order of preference:
      1) dedicated compatible hard-mask cache;
      2) compatible legacy cache, converted once;
      3) rebuild all four horizons from official TRAIN.
    """
    dedicated = hardmask_teacher_path(
        dataset_name,
        H,
        K,
    )

    if dedicated.exists():
        cache = np.load(
            dedicated
        )

        idx = cache[
            "source_idx"
        ]

        gain = cache[
            "source_gain"
        ]

        if (
            idx.shape
            ==
            (
                C,
                K,
            )
            and
            gain.shape
            ==
            (
                C,
                K,
            )
        ):
            return dedicated

        print(
            "Ignoring incompatible dedicated cache:",
            dedicated
        )

    # Try old cache without changing K.
    legacy = find_legacy_teacher_cache(
        dataset_name,
        H,
    )

    converted = _extract_legacy_endpoint(
        legacy,
        H,
        C,
        K,
    )

    if converted is not None:
        idx, gain = converted

        np.savez_compressed(
            dedicated,
            source_idx=idx,
            source_gain=gain,
            dataset=np.asarray(
                dataset_name
            ),
            pred_len=np.asarray(
                H,
                dtype=np.int64,
            ),
            channels=np.asarray(
                C,
                dtype=np.int64,
            ),
            topk=np.asarray(
                K,
                dtype=np.int64,
            ),
            source=np.asarray(
                "converted_legacy"
            ),
        )

        print(
            f"Converted compatible legacy teacher: "
            f"{dataset_name} H={H}"
        )

        return dedicated

    # Important: do not silently shrink K.
    print(
        f"Legacy teacher insufficient for "
        f"{dataset_name} H={H}, K={K}. "
        f"Rebuilding from official TRAIN."
    )

    rebuild_dataset_hardmask_teacher(
        dataset_name
    )

    if not dedicated.exists():
        raise RuntimeError(
            f"Teacher rebuild did not create {dedicated}"
        )

    return dedicated


def load_endpoint_teacher(
    dataset_name,
    H,
    C,
    K,
):
    path = ensure_hardmask_teacher_cache(
        dataset_name,
        H,
        C,
        K,
    )

    cache = np.load(
        path
    )

    idx = cache[
        "source_idx"
    ].astype(
        np.int64
    )

    gain = cache[
        "source_gain"
    ].astype(
        np.float32
    )

    if idx.shape != (
        C,
        K,
    ):
        raise RuntimeError(
            f"{dataset_name} H={H}: "
            f"source_idx shape {idx.shape}, "
            f"expected {(C,K)}"
        )

    if gain.shape != (
        C,
        K,
    ):
        raise RuntimeError(
            f"{dataset_name} H={H}: "
            f"source_gain shape {gain.shape}, "
            f"expected {(C,K)}"
        )

    return (
        idx,
        gain,
        path,
    )


def build_adaptive_endpoint_sources(
    dataset_name,
    H,
    C,
    K,
):
    """
    Dedicated cache is already guaranteed to contain exactly K unique
    valid sources for each target.
    """
    idx, gain, path = load_endpoint_teacher(
        dataset_name,
        H,
        C,
        K,
    )

    sources = np.zeros(
        (
            C,
            K,
        ),
        dtype=np.int64,
    )

    gains = np.zeros(
        (
            C,
            K,
        ),
        dtype=np.float32,
    )

    for i in range(C):
        ids, gs = _top_unique_valid_from_arrays(
            idx[
                i
            ],
            gain[
                i
            ],
            i,
            C,
            K,
        )

        if len(
            ids
        ) < K:
            raise RuntimeError(
                f"{dataset_name} H={H} target={i}: "
                f"dedicated cache has only {len(ids)} "
                f"unique sources, expected K={K}."
            )

        sources[
            i
        ] = ids

        gains[
            i
        ] = gs

    return (
        sources,
        gains,
        path,
    )


def build_shared_consensus_sources(
    dataset_name,
    C,
    K,
):
    """
    Horizon-invariant Borda-style consensus across endpoint rankings
    at H=96/192/336/720.
    """
    rankings = {}

    for H in ALL_HORIZONS:
        sources, gains, path = (
            build_adaptive_endpoint_sources(
                dataset_name,
                H,
                C,
                K,
            )
        )

        rankings[
            H
        ] = (
            sources,
            gains,
            path,
        )

    shared = np.zeros(
        (
            C,
            K,
        ),
        dtype=np.int64,
    )

    for i in range(C):
        score = {}
        freq = {}
        gain_sum = {}

        for H in ALL_HORIZONS:
            sources, gains, _ = (
                rankings[
                    H
                ]
            )

            for rank, (
                j,
                g,
            ) in enumerate(
                zip(
                    sources[
                        i
                    ].tolist(),
                    gains[
                        i
                    ].tolist(),
                )
            ):
                j = int(
                    j
                )

                borda = (
                    K
                    -
                    rank
                )

                score[
                    j
                ] = (
                    score.get(
                        j,
                        0.0,
                    )
                    +
                    float(
                        borda
                    )
                )

                freq[
                    j
                ] = (
                    freq.get(
                        j,
                        0
                    )
                    +
                    1
                )

                gain_sum[
                    j
                ] = (
                    gain_sum.get(
                        j,
                        0.0,
                    )
                    +
                    float(
                        g
                    )
                )

        candidates = list(
            score.keys()
        )

        candidates.sort(
            key=lambda j: (
                score[
                    j
                ],
                freq[
                    j
                ],
                gain_sum[
                    j
                ],
            ),
            reverse=True,
        )

        if len(
            candidates
        ) < K:
            raise RuntimeError(
                f"{dataset_name}, target={i}: "
                f"shared candidate union={len(candidates)} < K={K}"
            )

        shared[
            i
        ] = np.asarray(
            candidates[
                :K
            ],
            dtype=np.int64,
        )

    return (
        shared,
        rankings,
    )


def sources_to_allowed_mask(
    sources,
    C,
):
    """
    True = attention edge is allowed.

    Self edge + K selected source channels are always allowed.
    """
    sources = np.asarray(
        sources,
        dtype=np.int64,
    )

    K = sources.shape[
        1
    ]

    allowed = np.zeros(
        (
            C,
            C,
        ),
        dtype=bool,
    )

    for i in range(C):
        allowed[
            i,
            i
        ] = True

        allowed[
            i,
            sources[
                i
            ]
        ] = True

    row_counts = allowed.sum(
        axis=1
    )

    expected = (
        K
        +
        1
    )

    if not np.all(
        row_counts
        ==
        expected
    ):
        raise RuntimeError(
            f"Mask row count mismatch: "
            f"{np.unique(row_counts)} vs {expected}"
        )

    return allowed


def source_set_jaccard(
    A,
    B,
):
    vals = []

    for a, b in zip(
        A,
        B,
    ):
        sa = set(
            a.tolist()
        )

        sb = set(
            b.tolist()
        )

        vals.append(
            len(
                sa
                &
                sb
            )
            /
            max(
                1,
                len(
                    sa
                    |
                    sb
                ),
            )
        )

    return float(
        np.mean(
            vals
        )
    )


def build_condition_masks(
    dataset_name,
    H,
):
    C = DATASET_SPECS[
        dataset_name
    ][
        "channels"
    ]

    K = effective_topk(
        C
    )

    (
        shared_sources,
        all_rankings,
    ) = build_shared_consensus_sources(
        dataset_name,
        C,
        K,
    )

    (
        adaptive_sources,
        adaptive_gains,
        cache,
    ) = build_adaptive_endpoint_sources(
        dataset_name,
        H,
        C,
        K,
    )

    shared_mask = sources_to_allowed_mask(
        shared_sources,
        C,
    )

    adaptive_mask = sources_to_allowed_mask(
        adaptive_sources,
        C,
    )

    jaccard_value = source_set_jaccard(
        shared_sources,
        adaptive_sources,
    )

    return {
        "C": C,
        "K": K,
        "shared_sources": shared_sources,
        "adaptive_sources": adaptive_sources,
        "shared_mask": shared_mask,
        "adaptive_mask": adaptive_mask,
        "shared_adaptive_jaccard": jaccard_value,
        "adaptive_cache": str(
            cache
        ),
    }


# ---------------------------------------------------------------------
# Build / validate masks for all screening conditions.
# The first incompatible legacy cache will trigger a one-time rebuild
# for that dataset; subsequent conditions reuse the dedicated cache.
# ---------------------------------------------------------------------
mask_summary_rows = []

for d, H in CONDITIONS:
    print(
        "\n"
        +
        "-" * 88
    )

    print(
        f"Mask preparation: {d} H={H}"
    )

    info = build_condition_masks(
        d,
        H,
    )

    mask_summary_rows.append(
        {
            "dataset": d,
            "pred_len": H,
            "C": info[
                "C"
            ],
            "K": info[
                "K"
            ],
            "shared_adaptive_jaccard": info[
                "shared_adaptive_jaccard"
            ],
            "adaptive_cache": info[
                "adaptive_cache"
            ],
        }
    )

mask_summary = pd.DataFrame(
    mask_summary_rows
)

display(
    mask_summary.round(
        4
    )
)



----------------------------------------------------------------------------------------
Mask preparation: Electricity H=96

----------------------------------------------------------------------------------------
Mask preparation: Electricity H=192

----------------------------------------------------------------------------------------
Mask preparation: Electricity H=336

----------------------------------------------------------------------------------------
Mask preparation: Electricity H=720

----------------------------------------------------------------------------------------
Mask preparation: Weather H=96

----------------------------------------------------------------------------------------
Mask preparation: Weather H=192

----------------------------------------------------------------------------------------
Mask preparation: Weather H=336

----------------------------------------------------------------------------------------
Mask preparation: Weather H=720

---------

,dataset,pred_len,C,K,shared_adaptive_jaccard,adaptive_cache
0,Electricity,96,321,10,0.4412,/data/code/2026_08/results_itransformer_explic...
1,Electricity,192,321,10,0.5504,/data/code/2026_08/results_itransformer_explic...
2,Electricity,336,321,10,0.4867,/data/code/2026_08/results_itransformer_explic...
3,Electricity,720,321,10,0.4285,/data/code/2026_08/results_itransformer_explic...
4,Weather,96,21,10,0.7586,/data/code/2026_08/results_itransformer_explic...
5,Weather,192,21,10,0.7656,/data/code/2026_08/results_itransformer_explic...
6,Weather,336,21,10,0.7814,/data/code/2026_08/results_itransformer_explic...
7,Weather,720,21,10,0.6343,/data/code/2026_08/results_itransformer_explic...
8,Solar,96,137,10,0.6071,/data/code/2026_08/results_itransformer_explic...
9,Solar,192,137,10,0.6579,/data/code/2026_08/results_itransformer_explic...


## 8. Strong-backbone model with explicit last-block mask — exact base forward

This section implements the matched horizon-specific topology-transfer mechanism. Predictive-utility source sets are computed from training data and then imposed on the neural forecaster as specified in the paper.


In [10]:
class _FixedAttentionMask:
    """
    Minimal object compatible with TSL FullAttention.

    FullAttention reads:
        attn_mask.mask

    True = blocked
    False = allowed
    """
    def __init__(
        self,
        mask,
    ):
        self._mask = mask

    @property
    def mask(
        self,
    ):
        return self._mask


class _FixedMaskInnerAttention(
    nn.Module
):
    """
    Wrap ONLY the original last FullAttention module.

    Crucially:
      - Q/K/V projection remains in the original AttentionLayer.
      - FullAttention computation remains the original TSL implementation.
      - We replace only the incoming attn_mask with a fixed channel mask.
      - The rest of base.forward() is untouched.
    """

    def __init__(
        self,
        original_inner_attention,
        allowed_mask,
        channels,
    ):
        super().__init__()

        self.inner = (
            original_inner_attention
        )

        self.C = int(
            channels
        )

        allowed_mask = np.asarray(
            allowed_mask,
            dtype=bool,
        )

        if allowed_mask.shape != (
            self.C,
            self.C,
        ):
            raise ValueError(
                f"allowed_mask shape={allowed_mask.shape}, "
                f"expected {(self.C, self.C)}"
            )

        self.register_buffer(
            "allowed_mask",
            torch.as_tensor(
                allowed_mask,
                dtype=torch.bool,
            ),
        )

        if not hasattr(
            self.inner,
            "mask_flag",
        ):
            raise RuntimeError(
                "Expected TSL FullAttention with mask_flag."
            )

    def _make_mask(
        self,
        B,
        L,
        device,
    ):
        if L < self.C:
            raise RuntimeError(
                f"Token count L={L} < C={self.C}"
            )

        # Start with all interactions allowed.
        blocked = torch.zeros(
            (
                L,
                L,
            ),
            dtype=torch.bool,
            device=device,
        )

        # Channel tokens are the first C inverted-variable tokens.
        blocked[
            :self.C,
            :self.C,
        ] = ~self.allowed_mask.to(
            device=device
        )

        # [B,1,L,L] broadcasts over attention heads.
        blocked = (
            blocked[
                None,
                None,
                :,
                :,
            ]
            .expand(
                B,
                1,
                L,
                L,
            )
        )

        return _FixedAttentionMask(
            blocked
        )

    def forward(
        self,
        queries,
        keys,
        values,
        attn_mask,
        tau=None,
        delta=None,
    ):
        B = queries.shape[
            0
        ]

        L = queries.shape[
            1
        ]

        fixed_mask = self._make_mask(
            B,
            L,
            queries.device,
        )

        # iTransformer creates FullAttention(mask_flag=False).
        # For the sparse variants only, enable native mask handling.
        old_flag = self.inner.mask_flag

        self.inner.mask_flag = True

        try:
            out = self.inner(
                queries,
                keys,
                values,
                fixed_mask,
                tau=tau,
                delta=delta,
            )

        finally:
            # Restore original state so the wrapped object has no
            # side effect outside this forward call.
            self.inner.mask_flag = old_flag

        return out


class LastBlockSparseFineTune(
    nn.Module
):
    """
    v4: EXACT base.forward() execution.

    DenseFineTune
    -------------
    Calls:
        self.base(x, None, None, None)

    directly. No encoder splitting, no manual normalization,
    no manual projection, and no reconstructed last-layer path.

    SharedSparse / HorizonAdaptiveSparse
    ------------------------------------
    Also call the exact same base.forward().

    The ONLY intervention is:
        base.encoder.attn_layers[-1]
            .attention.inner_attention

    is wrapped so the original TSL FullAttention receives a fixed
    boolean channel mask.

    Frozen:
      - embedding
      - encoder blocks 1..L-1
      - final encoder norm

    Trainable:
      - last encoder block
      - forecast projection

    Thus SharedSparse vs HorizonAdaptiveSparse differs only in the
    fixed sparse topology.
    """

    def __init__(
        self,
        base_model,
        channels,
        allowed_mask=None,
    ):
        super().__init__()

        self.base = base_model

        self.C = int(
            channels
        )

        self.H = int(
            base_model.pred_len
        )

        self.last_layer = (
            self.base.encoder
            .attn_layers[-1]
        )

        # Freeze everything first.
        for p in self.base.parameters():
            p.requires_grad = False

        # Matched trainable subset.
        for p in self.last_layer.parameters():
            p.requires_grad = True

        for p in self.base.projection.parameters():
            p.requires_grad = True

        self.use_mask = (
            allowed_mask
            is not None
        )

        if self.use_mask:
            original_inner = (
                self.last_layer
                .attention
                .inner_attention
            )

            self.last_layer.attention.inner_attention = (
                _FixedMaskInnerAttention(
                    original_inner_attention=original_inner,
                    allowed_mask=allowed_mask,
                    channels=self.C,
                )
            )

        # Frozen model begins in eval mode.
        self.base.eval()

    def train(
        self,
        mode=True,
    ):
        """
        Keep the frozen prefix deterministic while training only
        the matched last block + projection.
        """
        super().train(
            mode
        )

        # Make every base submodule eval first.
        self.base.eval()

        # Enable training behavior only where parameters are trainable.
        self.last_layer.train(
            mode
        )

        self.base.projection.train(
            mode
        )

        return self

    def forward(
        self,
        x,
        return_attention=False,
    ):
        if return_attention:
            raise NotImplementedError(
                "Attention return is not needed in this screening."
            )

        # EXACT original iTransformer forward path.
        return self.base(
            x,
            None,
            None,
            None,
        )


## 9. Evaluation — full horizon and endpoint


In [11]:
@torch.no_grad()
def evaluate_base(
    base,
    loader,
    H,
):
    base.eval()

    full_sse = 0.0
    full_sae = 0.0
    full_n = 0

    endpoint_sse = 0.0
    endpoint_sae = 0.0
    endpoint_n = 0

    for x, y in loader:
        x = (
            x.float()
            .to(
                DEVICE,
                non_blocking=True,
            )
        )

        y = (
            y.float()
            .to(
                DEVICE,
                non_blocking=True,
            )
        )

        pred = base(
            x,
            None,
            None,
            None,
        )[
            :,
            -H:,
            :
        ]

        target = y[
            :,
            -H:,
            :
        ]

        err = (
            pred
            -
            target
        )

        full_sse += (
            err.square()
            .sum()
            .item()
        )

        full_sae += (
            err.abs()
            .sum()
            .item()
        )

        full_n += err.numel()

        end_err = err[
            :,
            -1,
            :
        ]

        endpoint_sse += (
            end_err.square()
            .sum()
            .item()
        )

        endpoint_sae += (
            end_err.abs()
            .sum()
            .item()
        )

        endpoint_n += (
            end_err.numel()
        )

    return {
        "full_mse": full_sse / full_n,
        "full_mae": full_sae / full_n,
        "endpoint_mse": (
            endpoint_sse
            /
            endpoint_n
        ),
        "endpoint_mae": (
            endpoint_sae
            /
            endpoint_n
        ),
    }


@torch.no_grad()
def evaluate_model(
    model,
    loader,
    H,
):
    model.eval()

    full_sse = 0.0
    full_sae = 0.0
    full_n = 0

    endpoint_sse = 0.0
    endpoint_sae = 0.0
    endpoint_n = 0

    for x, y in loader:
        x = (
            x.float()
            .to(
                DEVICE,
                non_blocking=True,
            )
        )

        y = (
            y.float()
            .to(
                DEVICE,
                non_blocking=True,
            )
        )

        pred = model(
            x
        )

        target = y[
            :,
            -H:,
            :
        ]

        err = (
            pred
            -
            target
        )

        full_sse += (
            err.square()
            .sum()
            .item()
        )

        full_sae += (
            err.abs()
            .sum()
            .item()
        )

        full_n += err.numel()

        end_err = err[
            :,
            -1,
            :
        ]

        endpoint_sse += (
            end_err.square()
            .sum()
            .item()
        )

        endpoint_sae += (
            end_err.abs()
            .sum()
            .item()
        )

        endpoint_n += (
            end_err.numel()
        )

    return {
        "full_mse": full_sse / full_n,
        "full_mae": full_sae / full_n,
        "endpoint_mse": (
            endpoint_sse
            /
            endpoint_n
        ),
        "endpoint_mae": (
            endpoint_sae
            /
            endpoint_n
        ),
    }


## 10. Matched fine-tuning protocol

This section implements the matched horizon-specific topology-transfer mechanism. Predictive-utility source sets are computed from training data and then imposed on the neural forecaster as specified in the paper.


In [12]:
def trainable_parameters(
    model,
):
    return [
        p
        for p in model.parameters()
        if p.requires_grad
    ]


def copy_trainable_state_to_cpu(
    model,
):
    return {
        "last_layer": {
            k: v.detach()
            .cpu()
            .clone()

            for k, v
            in model.last_layer
            .state_dict()
            .items()
        },

        "projection": {
            k: v.detach()
            .cpu()
            .clone()

            for k, v
            in model.base.projection
            .state_dict()
            .items()
        },
    }


def load_trainable_state(
    model,
    state,
):
    model.last_layer.load_state_dict(
        state[
            "last_layer"
        ],
        strict=True,
    )

    model.base.projection.load_state_dict(
        state[
            "projection"
        ],
        strict=True,
    )


def train_variant(
    model,
    train_loader,
    val_loader,
    H,
):
    params = trainable_parameters(
        model
    )

    optimizer = torch.optim.AdamW(
        params,
        lr=FINETUNE_LR,
        weight_decay=WEIGHT_DECAY,
    )

    # Evaluate masked/dense initialized model before any update.
    init_val = evaluate_model(
        model,
        val_loader,
        H,
    )

    best_val = float(
        init_val[
            "full_mse"
        ]
    )

    best_epoch = 0

    best_state = (
        copy_trainable_state_to_cpu(
            model
        )
    )

    bad = 0

    history = [
        {
            "epoch": 0,
            "train_mse": np.nan,
            "val_full_mse": init_val[
                "full_mse"
            ],
            "val_endpoint_mse": init_val[
                "endpoint_mse"
            ],
        }
    ]

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        model.train()

        train_sse = 0.0
        train_n = 0

        for x, y in train_loader:
            x = (
                x.float()
                .to(
                    DEVICE,
                    non_blocking=True,
                )
            )

            y = (
                y.float()
                .to(
                    DEVICE,
                    non_blocking=True,
                )
            )

            target = y[
                :,
                -H:,
                :
            ]

            optimizer.zero_grad(
                set_to_none=True
            )

            pred = model(
                x
            )

            loss = F.mse_loss(
                pred,
                target,
            )

            if not torch.isfinite(
                loss
            ):
                raise FloatingPointError(
                    "Non-finite training loss."
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                params,
                GRAD_CLIP,
            )

            optimizer.step()

            train_sse += (
                (
                    pred.detach()
                    -
                    target
                )
                .square()
                .sum()
                .item()
            )

            train_n += target.numel()

        val = evaluate_model(
            model,
            val_loader,
            H,
        )

        train_mse = (
            train_sse
            /
            train_n
        )

        history.append(
            {
                "epoch": epoch,
                "train_mse": (
                    train_mse
                ),
                "val_full_mse": (
                    val[
                        "full_mse"
                    ]
                ),
                "val_endpoint_mse": (
                    val[
                        "endpoint_mse"
                    ]
                ),
            }
        )

        print(
            f"epoch={epoch:02d} | "
            f"train={train_mse:.6f} | "
            f"val={val['full_mse']:.6f} | "
            f"end={val['endpoint_mse']:.6f}"
        )

        if (
            val[
                "full_mse"
            ]
            <
            best_val
        ):
            best_val = float(
                val[
                    "full_mse"
                ]
            )

            best_epoch = epoch

            best_state = (
                copy_trainable_state_to_cpu(
                    model
                )
            )

            bad = 0

        else:
            bad += 1

            if bad >= PATIENCE:
                print(
                    "Early stopping."
                )
                break

    load_trainable_state(
        model,
        best_state,
    )

    return (
        model,
        best_val,
        best_epoch,
        pd.DataFrame(
            history
        ),
    )


## 11. Pre-flight: exact state equality, same-instance forward identity, and mask efficacy

This section implements the matched horizon-specific topology-transfer mechanism. Predictive-utility source sets are computed from training data and then imposed on the neural forecaster as specified in the paper.


In [13]:
PF_DATASET = "ETTh1"
PF_H = 336

(
    _,
    train_pf,
    val_pf,
    test_pf,
    meta_pf,
) = prepare_datasets(
    PF_DATASET,
    PF_H,
)

(
    train_loader_pf,
    val_loader_pf,
    test_loader_pf,
) = make_loaders(
    train_pf,
    val_pf,
    test_pf,
    choose_batch_size(
        PF_DATASET,
        PF_H,
    ),
    SEED,
)

# ------------------------------------------------------------
# 1) Phase-1 checkpoint fidelity.
# ------------------------------------------------------------
base_pf = build_and_load_itransformer(
    PF_DATASET,
    PF_H,
)

stored_pf = load_phase1_result(
    PF_DATASET,
    PF_H,
)

base_test_pf = evaluate_base(
    base_pf,
    test_loader_pf,
    PF_H,
)

print(
    "Stored Phase-1 test MSE:",
    stored_pf[
        "test_mse"
    ]
)

print(
    "Reloaded checkpoint test MSE:",
    base_test_pf[
        "full_mse"
    ]
)

fidelity = abs(
    float(
        stored_pf[
            "test_mse"
        ]
    )
    -
    float(
        base_test_pf[
            "full_mse"
        ]
    )
)

print(
    "Checkpoint fidelity abs diff:",
    fidelity
)

assert fidelity < 2e-5


# ------------------------------------------------------------
# 2) Build masks and verify exactly self + K allowed edges.
# ------------------------------------------------------------
mask_pf = build_condition_masks(
    PF_DATASET,
    PF_H,
)

print(
    "C:",
    mask_pf[
        "C"
    ],
    "| K:",
    mask_pf[
        "K"
    ],
    "| Shared/Adaptive Jaccard:",
    mask_pf[
        "shared_adaptive_jaccard"
    ],
)

shared_row_counts = (
    mask_pf[
        "shared_mask"
    ].sum(
        axis=1
    )
)

adaptive_row_counts = (
    mask_pf[
        "adaptive_mask"
    ].sum(
        axis=1
    )
)

assert np.all(
    shared_row_counts
    ==
    mask_pf[
        "K"
    ]
    +
    1
)

assert np.all(
    adaptive_row_counts
    ==
    mask_pf[
        "K"
    ]
    +
    1
)


# ------------------------------------------------------------
# 3) Independently loaded checkpoint states must be EXACT.
# ------------------------------------------------------------
base_identity = build_and_load_itransformer(
    PF_DATASET,
    PF_H,
)

dense_pf = LastBlockSparseFineTune(
    build_and_load_itransformer(
        PF_DATASET,
        PF_H,
    ),
    channels=mask_pf[
        "C"
    ],
    allowed_mask=None,
).to(
    DEVICE
)

base_identity.eval()
dense_pf.eval()

state_a = base_identity.state_dict()
state_b = dense_pf.base.state_dict()

if list(
    state_a.keys()
) != list(
    state_b.keys()
):
    raise AssertionError(
        "Checkpoint state_dict keys differ."
    )

max_state_diff = 0.0
state_mismatch_count = 0

for key in state_a:
    a = state_a[
        key
    ].detach().cpu()

    b = state_b[
        key
    ].detach().cpu()

    if not torch.equal(
        a,
        b,
    ):
        state_mismatch_count += 1

        if (
            torch.is_floating_point(
                a
            )
            or
            torch.is_complex(
                a
            )
        ):
            diff = (
                a
                -
                b
            ).abs().max().item()

            max_state_diff = max(
                max_state_diff,
                float(
                    diff
                ),
            )

        else:
            max_state_diff = float(
                "inf"
            )

print(
    "Independent checkpoint state mismatches:",
    state_mismatch_count
)

print(
    "Independent checkpoint max state diff:",
    max_state_diff
)

if state_mismatch_count != 0:
    raise AssertionError(
        "Independently loaded checkpoints do not have identical state tensors."
    )


# ------------------------------------------------------------
# 4) SAME-INSTANCE direct base.forward vs wrapper forward.
# ------------------------------------------------------------
x_pf, _ = next(
    iter(
        test_loader_pf
    )
)

x_pf = (
    x_pf.float()
    .to(
        DEVICE
    )
)

dense_pf.eval()

with torch.no_grad():
    # Both calls use the exact SAME parameter tensors and module instance.
    p_direct_same_instance = dense_pf.base(
        x_pf,
        None,
        None,
        None,
    )

    p_wrapper = dense_pf(
        x_pf
    )

same_instance_error = (
    p_direct_same_instance
    -
    p_wrapper
).abs().max().item()

print(
    "Dense same-instance base.forward vs wrapper max abs error:",
    same_instance_error
)

if same_instance_error >= 1e-7:
    raise AssertionError(
        f"Wrapper changes Dense forward on the same model instance: "
        f"max abs error={same_instance_error:.6e}."
    )


# ------------------------------------------------------------
# 5) Cross-instance output difference is diagnostic only.
#    State equality above is the actual correctness criterion.
# ------------------------------------------------------------
with torch.no_grad():
    p_independent = base_identity(
        x_pf,
        None,
        None,
        None,
    )

cross_instance_error = (
    p_independent
    -
    p_wrapper
).abs().max().item()

print(
    "Independent-model output max abs diff (diagnostic only):",
    cross_instance_error
)


# ------------------------------------------------------------
# 6) Shared/Adaptive sparse variants:
#    same trainable tensors at initialization.
# ------------------------------------------------------------
shared_pf = LastBlockSparseFineTune(
    build_and_load_itransformer(
        PF_DATASET,
        PF_H,
    ),
    channels=mask_pf[
        "C"
    ],
    allowed_mask=mask_pf[
        "shared_mask"
    ],
).to(
    DEVICE
)

adaptive_pf = LastBlockSparseFineTune(
    build_and_load_itransformer(
        PF_DATASET,
        PF_H,
    ),
    channels=mask_pf[
        "C"
    ],
    allowed_mask=mask_pf[
        "adaptive_mask"
    ],
).to(
    DEVICE
)

n_dense = sum(
    p.numel()
    for p in dense_pf.parameters()
    if p.requires_grad
)

n_shared = sum(
    p.numel()
    for p in shared_pf.parameters()
    if p.requires_grad
)

n_adaptive = sum(
    p.numel()
    for p in adaptive_pf.parameters()
    if p.requires_grad
)

print(
    "Trainable params:",
    n_dense,
    n_shared,
    n_adaptive
)

assert (
    n_dense
    ==
    n_shared
    ==
    n_adaptive
)

dense_trainable = [
    p.detach()
    .cpu()
    .clone()

    for p in trainable_parameters(
        dense_pf
    )
]

shared_trainable = [
    p.detach()
    .cpu()
    .clone()

    for p in trainable_parameters(
        shared_pf
    )
]

adaptive_trainable = [
    p.detach()
    .cpu()
    .clone()

    for p in trainable_parameters(
        adaptive_pf
    )
]

assert (
    len(
        dense_trainable
    )
    ==
    len(
        shared_trainable
    )
    ==
    len(
        adaptive_trainable
    )
)

for (
    pdense,
    pshared,
    padaptive,
) in zip(
    dense_trainable,
    shared_trainable,
    adaptive_trainable,
):
    if not torch.equal(
        pdense,
        pshared,
    ):
        raise AssertionError(
            "Dense and SharedSparse trainable initialization differs."
        )

    if not torch.equal(
        pdense,
        padaptive,
    ):
        raise AssertionError(
            "Dense and AdaptiveSparse trainable initialization differs."
        )


# ------------------------------------------------------------
# 7) Sparse intervention must actually change output.
# ------------------------------------------------------------
shared_pf.eval()
adaptive_pf.eval()

with torch.no_grad():
    p_dense = dense_pf(
        x_pf
    )

    p_shared = shared_pf(
        x_pf
    )

    p_adaptive = adaptive_pf(
        x_pf
    )

shared_vs_dense = (
    p_shared
    -
    p_dense
).abs().max().item()

adaptive_vs_dense = (
    p_adaptive
    -
    p_dense
).abs().max().item()

shared_vs_adaptive = (
    p_shared
    -
    p_adaptive
).abs().max().item()

print(
    "SharedSparse vs Dense max abs diff:",
    shared_vs_dense
)

print(
    "AdaptiveSparse vs Dense max abs diff:",
    adaptive_vs_dense
)

print(
    "SharedSparse vs AdaptiveSparse max abs diff:",
    shared_vs_adaptive
)

if shared_vs_dense <= 1e-8:
    raise AssertionError(
        "Shared sparse mask appears to have no effect."
    )

if adaptive_vs_dense <= 1e-8:
    raise AssertionError(
        "Adaptive sparse mask appears to have no effect."
    )

print(
    "\nPre-flight: PASSED"
)


del (
    train_pf,
    val_pf,
    test_pf,
    train_loader_pf,
    val_loader_pf,
    test_loader_pf,
    base_pf,
    base_identity,
    dense_pf,
    shared_pf,
    adaptive_pf,
    x_pf,
    p_direct_same_instance,
    p_wrapper,
    p_independent,
    p_dense,
    p_shared,
    p_adaptive,
    dense_trainable,
    shared_trainable,
    adaptive_trainable,
    state_a,
    state_b,
)

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


Stored Phase-1 test MSE: 0.4768634684235003
Reloaded checkpoint test MSE: 0.4768634684235003
Checkpoint fidelity abs diff: 0.0
C: 7 | K: 3 | Shared/Adaptive Jaccard: 0.7428571428571429
Independent checkpoint state mismatches: 0
Independent checkpoint max state diff: 0.0
Dense same-instance base.forward vs wrapper max abs error: 0.0
Independent-model output max abs diff (diagnostic only): 0.0010225176811218262
Trainable params: 1750352 1750352 1750352
SharedSparse vs Dense max abs diff: 0.8624730110168457
AdaptiveSparse vs Dense max abs diff: 0.932470977306366
SharedSparse vs AdaptiveSparse max abs diff: 0.1245923638343811

Pre-flight: PASSED


## 12. Ridge reference — diagnostic only, never used for training


In [14]:
if not RIDGE_REFERENCE.exists():
    raise FileNotFoundError(
        RIDGE_REFERENCE
    )

ridge_ref = pd.read_csv(
    RIDGE_REFERENCE
)

ridge_ref = ridge_ref[
    [
        "dataset",
        "horizon",
        "always_adaptive_gain_%",
        "oracle_gain_%",
    ]
].copy()

ridge_ref = ridge_ref.rename(
    columns={
        "horizon":
        "pred_len",

        "always_adaptive_gain_%":
        "ridge_adaptive_gain_%",
    }
)

screening_ref = pd.DataFrame(
    CONDITIONS,
    columns=[
        "dataset",
        "pred_len",
    ],
).merge(
    ridge_ref,
    on=[
        "dataset",
        "pred_len",
    ],
    how="left",
    validate="one_to_one",
)

if screening_ref[
    "ridge_adaptive_gain_%"
].isna().any():
    raise RuntimeError(
        "Missing Ridge reference for a screening condition."
    )

display(
    screening_ref.round(
        4
    )
)

print(
    "\nThis table is used only for post-hoc transfer diagnosis."
)
print(
    "No Ridge test gain enters the model or mask construction."
)


,dataset,pred_len,ridge_adaptive_gain_%,oracle_gain_%
0,Electricity,96,-0.0445,2.9525
1,Electricity,192,1.4992,3.8714
2,Electricity,336,-2.1514,2.2604
3,Electricity,720,0.6137,3.7273
4,Weather,96,-1.3256,0.5105
5,Weather,192,0.8131,1.5396
6,Weather,336,0.7264,1.1287
7,Weather,720,-3.7116,1.6260
8,Solar,96,-1.1805,0.4771
9,Solar,192,-1.5726,0.2140



This table is used only for post-hoc transfer diagnosis.
No Ridge test gain enters the model or mask construction.


## 13. Run matched strong-backbone screening


In [ ]:
def variant_dir(
    dataset_name,
    H,
    variant,
):
    path = (
        OUTPUT_DIR
        / dataset_name
        / f"H{H}"
        / variant
    )

    path.mkdir(
        parents=True,
        exist_ok=True,
    )

    return path


def result_path(
    dataset_name,
    H,
    variant,
):
    return (
        variant_dir(
            dataset_name,
            H,
            variant,
        )
        / "result.csv"
    )


def history_path(
    dataset_name,
    H,
    variant,
):
    return (
        variant_dir(
            dataset_name,
            H,
            variant,
        )
        / "history.csv"
    )


def load_completed(
    dataset_name,
    H,
    variant,
):
    # Prefer the all-20 output. If absent, reuse the original 8-condition
    # result and copy it into the new result tree.
    new_path = result_path(dataset_name, H, variant)
    old_path = (
        ORIGINAL_OUTPUT_DIR
        / dataset_name
        / f"H{H}"
        / variant
        / "result.csv"
    )

    for path in [new_path, old_path]:
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path)
            if len(df) != 1:
                continue
            row = df.iloc[0].to_dict()
            required = ["test_full_mse", "test_endpoint_mse"]
            if not all(np.isfinite(row[k]) for k in required):
                continue
            if path == old_path and path != new_path:
                pd.DataFrame([row]).to_csv(new_path, index=False)
                print("Reused original 8-condition result:", old_path)
            return row
        except Exception:
            continue
    return None


all_results = []
failures = []

total_runs = (
    len(
        CONDITIONS
    )
    *
    len(
        VARIANTS
    )
)

run_idx = 0

for (
    dataset_name,
    H,
) in CONDITIONS:
    print(
        "\n"
        +
        "#"
        *
        120
    )

    print(
        f"{dataset_name} H={H}"
    )

    print(
        "#"
        *
        120
    )

    (
        _,
        train_ds,
        val_ds,
        test_ds,
        meta,
    ) = prepare_datasets(
        dataset_name,
        H,
    )

    batch_size = choose_batch_size(
        dataset_name,
        H,
    )

    condition_masks = build_condition_masks(
        dataset_name,
        H,
    )

    # Evaluation loaders for the frozen Phase-1 reference.
    # Training loaders are recreated per variant below.
    (
        _reference_train_loader,
        val_loader,
        test_loader,
    ) = make_loaders(
        train_ds,
        val_ds,
        test_ds,
        batch_size,
        SEED,
    )

    base = build_and_load_itransformer(
        dataset_name,
        H,
    )

    base_val = evaluate_base(
        base,
        val_loader,
        H,
    )

    base_test = evaluate_base(
        base,
        test_loader,
        H,
    )

    stored = load_phase1_result(
        dataset_name,
        H,
    )

    checkpoint_fidelity = abs(
        float(
            stored[
                "test_mse"
            ]
        )
        -
        float(
            base_test[
                "full_mse"
            ]
        )
    )

    print(
        f"C={meta['C']} | "
        f"K={condition_masks['K']} | "
        f"batch={batch_size} | "
        f"mask Jaccard="
        f"{condition_masks['shared_adaptive_jaccard']:.4f}"
    )

    print(
        f"Phase-1 test MSE="
        f"{base_test['full_mse']:.6f} | "
        f"endpoint="
        f"{base_test['endpoint_mse']:.6f} | "
        f"fidelity="
        f"{checkpoint_fidelity:.3e}"
    )

    for variant in VARIANTS:
        run_idx += 1

        print(
            "\n"
            +
            "="
            *
            120
        )

        print(
            f"[{run_idx}/{total_runs}] "
            f"{dataset_name} H={H} | {variant}"
        )

        print(
            "="
            *
            120
        )

        if RESUME:
            old = load_completed(
                dataset_name,
                H,
                variant,
            )

            if old is not None:
                print(
                    "Reusing completed result."
                )

                all_results.append(
                    old
                )

                continue

        try:
            set_seed(
                SEED
            )

            # CRITICAL matched-training fix:
            # recreate the DataLoaders with the SAME generator seed
            # for every variant so mini-batch order is identical.
            (
                train_loader_variant,
                val_loader_variant,
                test_loader_variant,
            ) = make_loaders(
                train_ds,
                val_ds,
                test_ds,
                batch_size,
                SEED,
            )

            base_variant = (
                build_and_load_itransformer(
                    dataset_name,
                    H,
                )
            )

            if variant == "DenseFineTune":
                mask = None

            elif variant == "SharedSparse":
                mask = condition_masks[
                    "shared_mask"
                ]

            elif (
                variant
                ==
                "HorizonAdaptiveSparse"
            ):
                mask = condition_masks[
                    "adaptive_mask"
                ]

            else:
                raise ValueError(
                    variant
                )

            model = (
                LastBlockSparseFineTune(
                    base_variant,
                    channels=meta[
                        "C"
                    ],
                    allowed_mask=mask,
                )
                .to(
                    DEVICE
                )
            )

            trainable = sum(
                p.numel()
                for p in model.parameters()
                if p.requires_grad
            )

            start = time.time()

            (
                model,
                best_val,
                best_epoch,
                history,
            ) = train_variant(
                model,
                train_loader_variant,
                val_loader_variant,
                H,
            )

            test = evaluate_model(
                model,
                test_loader_variant,
                H,
            )

            full_gain_vs_base = (
                100.0
                *
                (
                    base_test[
                        "full_mse"
                    ]
                    -
                    test[
                        "full_mse"
                    ]
                )
                /
                base_test[
                    "full_mse"
                ]
            )

            endpoint_gain_vs_base = (
                100.0
                *
                (
                    base_test[
                        "endpoint_mse"
                    ]
                    -
                    test[
                        "endpoint_mse"
                    ]
                )
                /
                base_test[
                    "endpoint_mse"
                ]
            )

            result = {
                "dataset": dataset_name,
                "pred_len": H,
                "variant": variant,

                "C": meta[
                    "C"
                ],
                "K": condition_masks[
                    "K"
                ],

                "shared_adaptive_jaccard": (
                    condition_masks[
                        "shared_adaptive_jaccard"
                    ]
                ),

                "batch_size": batch_size,
                "new_trainable_params": (
                    trainable
                ),

                "checkpoint_fidelity_abs_diff": (
                    checkpoint_fidelity
                ),

                "base_val_full_mse": (
                    base_val[
                        "full_mse"
                    ]
                ),

                "base_test_full_mse": (
                    base_test[
                        "full_mse"
                    ]
                ),

                "base_test_full_mae": (
                    base_test[
                        "full_mae"
                    ]
                ),

                "base_test_endpoint_mse": (
                    base_test[
                        "endpoint_mse"
                    ]
                ),

                "base_test_endpoint_mae": (
                    base_test[
                        "endpoint_mae"
                    ]
                ),

                "best_val_full_mse": (
                    best_val
                ),

                "best_epoch": (
                    best_epoch
                ),

                "test_full_mse": (
                    test[
                        "full_mse"
                    ]
                ),

                "test_full_mae": (
                    test[
                        "full_mae"
                    ]
                ),

                "test_endpoint_mse": (
                    test[
                        "endpoint_mse"
                    ]
                ),

                "test_endpoint_mae": (
                    test[
                        "endpoint_mae"
                    ]
                ),

                "full_mse_gain_vs_base_%": (
                    full_gain_vs_base
                ),

                "endpoint_mse_gain_vs_base_%": (
                    endpoint_gain_vs_base
                ),

                "val_improves_base": int(
                    best_val
                    <
                    base_val[
                        "full_mse"
                    ]
                ),

                "elapsed_sec": (
                    time.time()
                    -
                    start
                ),
            }

            pd.DataFrame(
                [
                    result
                ]
            ).to_csv(
                result_path(
                    dataset_name,
                    H,
                    variant,
                ),
                index=False,
            )

            history.to_csv(
                history_path(
                    dataset_name,
                    H,
                    variant,
                ),
                index=False,
            )

            all_results.append(
                result
            )

            print(
                f"test MSE="
                f"{test['full_mse']:.6f} | "
                f"gain={full_gain_vs_base:+.3f}%"
            )

            print(
                f"endpoint MSE="
                f"{test['endpoint_mse']:.6f} | "
                f"gain={endpoint_gain_vs_base:+.3f}%"
            )

            del (
                model,
                base_variant,
                train_loader_variant,
                val_loader_variant,
                test_loader_variant,
            )

        except Exception as e:
            failures.append(
                {
                    "dataset": dataset_name,
                    "pred_len": H,
                    "variant": variant,
                    "error": repr(
                        e
                    ),
                }
            )

            print(
                "FAILED:",
                repr(
                    e
                )
            )

        gc.collect()

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    del (
        train_ds,
        val_ds,
        test_ds,
        _reference_train_loader,
        val_loader,
        test_loader,
        base,
    )

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


results_df = pd.DataFrame(
    all_results
)

failures_df = pd.DataFrame(
    failures
)

results_df.to_csv(
    OUTPUT_DIR
    / "strong_sparse_mask_screening_results.csv",
    index=False,
)

failures_df.to_csv(
    OUTPUT_DIR
    / "failures.csv",
    index=False,
)

print(
    "\nCompleted:",
    len(
        results_df
    ),
    "/",
    total_runs
)

print(
    "Failures:",
    len(
        failures_df
    )
)



########################################################################################################################
Electricity H=96
########################################################################################################################
C=321 | K=10 | batch=16 | mask Jaccard=0.4412
Phase-1 test MSE=0.151283 | endpoint=0.162443 | fidelity=0.000e+00

[1/60] Electricity H=96 | DenseFineTune
epoch=01 | train=0.133983 | val=0.129469 | end=0.133211
epoch=02 | train=0.133087 | val=0.129372 | end=0.134310
epoch=03 | train=0.132372 | val=0.128621 | end=0.133686
epoch=04 | train=0.131734 | val=0.128384 | end=0.133221
epoch=05 | train=0.131154 | val=0.128528 | end=0.133565
epoch=06 | train=0.130665 | val=0.128576 | end=0.133087
epoch=07 | train=0.130173 | val=0.127959 | end=0.132904
epoch=08 | train=0.129724 | val=0.128184 | end=0.132659
test MSE=0.149827 | gain=+0.962%
endpoint MSE=0.162508 | gain=-0.040%

[2/60] Electricity H=96 | SharedSparse
epoch=01 | train=0.141370 | 

## 14. Main result table


In [ ]:
if len(
    results_df
) == 0:
    raise RuntimeError(
        "No completed results."
    )

display_cols = [
    "dataset",
    "pred_len",
    "variant",
    "K",
    "shared_adaptive_jaccard",
    "base_test_full_mse",
    "test_full_mse",
    "full_mse_gain_vs_base_%",
    "base_test_endpoint_mse",
    "test_endpoint_mse",
    "endpoint_mse_gain_vs_base_%",
    "best_epoch",
    "val_improves_base",
]

display(
    results_df[
        display_cols
    ]
    .sort_values(
        [
            "dataset",
            "pred_len",
            "variant",
        ]
    )
    .round(
        6
    )
)


## 15. Decisive paired comparison: HorizonAdaptiveSparse vs SharedSparse


In [ ]:
full_pivot = (
    results_df
    .pivot_table(
        index=[
            "dataset",
            "pred_len",
        ],
        columns="variant",
        values="test_full_mse",
        aggfunc="first",
    )
    .reset_index()
)

endpoint_pivot = (
    results_df
    .pivot_table(
        index=[
            "dataset",
            "pred_len",
        ],
        columns="variant",
        values="test_endpoint_mse",
        aggfunc="first",
    )
    .reset_index()
)

paired = full_pivot.merge(
    endpoint_pivot,
    on=[
        "dataset",
        "pred_len",
    ],
    suffixes=(
        "_full",
        "_endpoint",
    ),
    validate="one_to_one",
)

required_variants = set(
    VARIANTS
)

for variant in required_variants:
    if (
        f"{variant}_full"
        not in paired.columns
    ):
        raise RuntimeError(
            f"Missing variant: {variant}"
        )


paired[
    "adaptive_vs_shared_full_gain_%"
] = (
    100.0
    *
    (
        paired[
            "SharedSparse_full"
        ]
        -
        paired[
            "HorizonAdaptiveSparse_full"
        ]
    )
    /
    paired[
        "SharedSparse_full"
    ]
)


paired[
    "adaptive_vs_shared_endpoint_gain_%"
] = (
    100.0
    *
    (
        paired[
            "SharedSparse_endpoint"
        ]
        -
        paired[
            "HorizonAdaptiveSparse_endpoint"
        ]
    )
    /
    paired[
        "SharedSparse_endpoint"
    ]
)


paired[
    "adaptive_beats_shared_full"
] = (
    paired[
        "adaptive_vs_shared_full_gain_%"
    ]
    >
    0
)


paired[
    "adaptive_beats_shared_endpoint"
] = (
    paired[
        "adaptive_vs_shared_endpoint_gain_%"
    ]
    >
    0
)


paired = paired.merge(
    screening_ref[
        [
            "dataset",
            "pred_len",
            "ridge_adaptive_gain_%",
        ]
    ],
    on=[
        "dataset",
        "pred_len",
    ],
    how="left",
    validate="one_to_one",
)


paired[
    "ridge_sign"
] = np.sign(
    paired[
        "ridge_adaptive_gain_%"
    ]
)


paired[
    "strong_full_sign"
] = np.sign(
    paired[
        "adaptive_vs_shared_full_gain_%"
    ]
)


paired[
    "strong_endpoint_sign"
] = np.sign(
    paired[
        "adaptive_vs_shared_endpoint_gain_%"
    ]
)


paired[
    "full_sign_agreement"
] = (
    paired[
        "ridge_sign"
    ]
    ==
    paired[
        "strong_full_sign"
    ]
)


paired[
    "endpoint_sign_agreement"
] = (
    paired[
        "ridge_sign"
    ]
    ==
    paired[
        "strong_endpoint_sign"
    ]
)


display(
    paired[
        [
            "dataset",
            "pred_len",
            "ridge_adaptive_gain_%",
            "adaptive_vs_shared_full_gain_%",
            "adaptive_vs_shared_endpoint_gain_%",
            "adaptive_beats_shared_full",
            "adaptive_beats_shared_endpoint",
            "full_sign_agreement",
            "endpoint_sign_agreement",
        ]
    ]
    .round(
        5
    )
)


paired.to_csv(
    OUTPUT_DIR
    / "adaptive_vs_shared_paired.csv",
    index=False,
)


## 16. Does the Ridge diagnostic transfer to the strong backbone?


In [ ]:
rho_full = float(
    paired[
        [
            "ridge_adaptive_gain_%",
            "adaptive_vs_shared_full_gain_%",
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1
    ]
)


rho_endpoint = float(
    paired[
        [
            "ridge_adaptive_gain_%",
            "adaptive_vs_shared_endpoint_gain_%",
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1
    ]
)


full_sign_agree = float(
    paired[
        "full_sign_agreement"
    ].mean()
)


endpoint_sign_agree = float(
    paired[
        "endpoint_sign_agreement"
    ].mean()
)


print(
    "Spearman Ridge -> strong full-horizon adaptive advantage:",
    round(
        rho_full,
        3,
    )
)


print(
    "Spearman Ridge -> strong endpoint adaptive advantage:",
    round(
        rho_endpoint,
        3,
    )
)


print(
    "Full-horizon sign agreement:",
    f"{full_sign_agree:.3f}"
)


print(
    "Endpoint sign agreement:",
    f"{endpoint_sign_agree:.3f}"
)


print(
    "\nAdaptiveSparse > SharedSparse:"
)


print(
    "  full horizon:",
    int(
        paired[
            "adaptive_beats_shared_full"
        ].sum()
    ),
    "/",
    len(
        paired
    ),
)


print(
    "  endpoint:",
    int(
        paired[
            "adaptive_beats_shared_endpoint"
        ].sum()
    ),
    "/",
    len(
        paired
    ),
)


## 17. Positive-reference vs negative-reference controls


In [ ]:
paired[
    "ridge_reference_group"
] = np.where(
    paired[
        "ridge_adaptive_gain_%"
    ]
    >
    0,
    "RidgePositive",
    "RidgeNegative",
)


control_summary = (
    paired
    .groupby(
        "ridge_reference_group",
        as_index=False,
    )
    .agg(
        n_conditions=(
            "dataset",
            "size",
        ),

        mean_ridge_gain=(
            "ridge_adaptive_gain_%",
            "mean",
        ),

        mean_strong_full_gain=(
            "adaptive_vs_shared_full_gain_%",
            "mean",
        ),

        mean_strong_endpoint_gain=(
            "adaptive_vs_shared_endpoint_gain_%",
            "mean",
        ),

        full_positive_fraction=(
            "adaptive_beats_shared_full",
            "mean",
        ),

        endpoint_positive_fraction=(
            "adaptive_beats_shared_endpoint",
            "mean",
        ),
    )
)


display(
    control_summary.round(
        4
    )
)


control_summary.to_csv(
    OUTPUT_DIR
    / "ridge_positive_negative_transfer_summary.csv",
    index=False,
)


## 18. Dense control: is sparsity itself useful?


In [ ]:
dense_compare = results_df.pivot_table(
    index=[
        "dataset",
        "pred_len",
    ],
    columns="variant",
    values=[
        "test_full_mse",
        "test_endpoint_mse",
    ],
    aggfunc="first",
)


rows = []

for idx, row in dense_compare.iterrows():
    dataset_name, H = idx

    dense_full = row[
        (
            "test_full_mse",
            "DenseFineTune",
        )
    ]

    shared_full = row[
        (
            "test_full_mse",
            "SharedSparse",
        )
    ]

    adaptive_full = row[
        (
            "test_full_mse",
            "HorizonAdaptiveSparse",
        )
    ]

    dense_endpoint = row[
        (
            "test_endpoint_mse",
            "DenseFineTune",
        )
    ]

    shared_endpoint = row[
        (
            "test_endpoint_mse",
            "SharedSparse",
        )
    ]

    adaptive_endpoint = row[
        (
            "test_endpoint_mse",
            "HorizonAdaptiveSparse",
        )
    ]

    rows.append(
        {
            "dataset": dataset_name,
            "pred_len": H,

            "shared_vs_dense_full_gain_%": (
                100.0
                *
                (
                    dense_full
                    -
                    shared_full
                )
                /
                dense_full
            ),

            "adaptive_vs_dense_full_gain_%": (
                100.0
                *
                (
                    dense_full
                    -
                    adaptive_full
                )
                /
                dense_full
            ),

            "shared_vs_dense_endpoint_gain_%": (
                100.0
                *
                (
                    dense_endpoint
                    -
                    shared_endpoint
                )
                /
                dense_endpoint
            ),

            "adaptive_vs_dense_endpoint_gain_%": (
                100.0
                *
                (
                    dense_endpoint
                    -
                    adaptive_endpoint
                )
                /
                dense_endpoint
            ),
        }
    )


dense_control = pd.DataFrame(
    rows
)


display(
    dense_control.round(
        5
    )
)


dense_control.to_csv(
    OUTPUT_DIR
    / "sparse_vs_dense_finetune.csv",
    index=False,
)


## 19. Mask-change magnitude vs strong-backbone effect


In [ ]:
mask_analysis = (
    paired.merge(
        mask_summary[
            [
                "dataset",
                "pred_len",
                "K",
                "shared_adaptive_jaccard",
            ]
        ],
        on=[
            "dataset",
            "pred_len",
        ],
        how="left",
        validate="one_to_one",
    )
)


mask_analysis[
    "source_set_change"
] = (
    1.0
    -
    mask_analysis[
        "shared_adaptive_jaccard"
    ]
)


rho_mask_full = float(
    mask_analysis[
        [
            "source_set_change",
            "adaptive_vs_shared_full_gain_%",
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1
    ]
)


rho_mask_endpoint = float(
    mask_analysis[
        [
            "source_set_change",
            "adaptive_vs_shared_endpoint_gain_%",
        ]
    ]
    .corr(
        method="spearman"
    )
    .iloc[
        0,
        1
    ]
)


print(
    "Spearman source-set change -> full gain:",
    round(
        rho_mask_full,
        3,
    )
)


print(
    "Spearman source-set change -> endpoint gain:",
    round(
        rho_mask_endpoint,
        3,
    )
)


display(
    mask_analysis[
        [
            "dataset",
            "pred_len",
            "K",
            "shared_adaptive_jaccard",
            "source_set_change",
            "adaptive_vs_shared_full_gain_%",
            "adaptive_vs_shared_endpoint_gain_%",
        ]
    ]
    .round(
        5
    )
)


## 20. Automatic decision


In [ ]:
n = len(paired)

n_full = int(paired["adaptive_beats_shared_full"].sum())
n_endpoint = int(paired["adaptive_beats_shared_endpoint"].sum())
n_full_sign = int(paired["full_sign_agreement"].sum())
n_endpoint_sign = int(paired["endpoint_sign_agreement"].sum())
mean_full = float(paired["adaptive_vs_shared_full_gain_%"].mean())
mean_endpoint = float(paired["adaptive_vs_shared_endpoint_gain_%"].mean())
median_full = float(paired["adaptive_vs_shared_full_gain_%"].median())
median_endpoint = float(paired["adaptive_vs_shared_endpoint_gain_%"].median())

print("=" * 88)
print("ALL-20 HARD-MASK TRANSFER SUMMARY")
print("=" * 88)
print(f"Conditions: {n}")
print(f"Adaptive > Shared, full:     {n_full}/{n} ({n_full/max(n,1):.1%})")
print(f"Adaptive > Shared, endpoint: {n_endpoint}/{n} ({n_endpoint/max(n,1):.1%})")
print(f"Mean full advantage:         {mean_full:+.4f}%")
print(f"Median full advantage:       {median_full:+.4f}%")
print(f"Mean endpoint advantage:     {mean_endpoint:+.4f}%")
print(f"Median endpoint advantage:   {median_endpoint:+.4f}%")
print(f"Ridge/full Spearman:         {rho_full:+.3f}")
print(f"Ridge/endpoint Spearman:     {rho_endpoint:+.3f}")
print(f"Ridge/full sign agreement:   {n_full_sign}/{n}")
print(f"Ridge/end sign agreement:    {n_endpoint_sign}/{n}")

print("\nInterpretation guide:")
print("- If the 20-condition result remains near-zero in mean gain and weak in sign/rank agreement,")
print("  it substantially strengthens the paper's 'utility does not reliably transfer' claim.")
print("- If Adaptive becomes consistently positive or strongly correlated with the controlled gain,")
print("  revise the main claim rather than hiding the expanded result.")


In [23]:
from pathlib import Path
import pandas as pd
from datetime import datetime

BASE = Path(
    "/data/code/2026_08/"
    "results_itransformer_explicit_sparse_horizon_mask_all20"
)

datasets = ["Electricity", "Weather", "Solar", "ETTh1", "ETTm1"]
horizons = [96, 192, 336, 720]
variants = ["DenseFineTune", "SharedSparse", "HorizonAdaptiveSparse"]

files = sorted(BASE.glob("*/H*/*/result.csv"))

print(f"Completed result files: {len(files)}/60")

completed = set()
for p in files:
    dataset = p.parts[-4]
    H = int(p.parts[-3][1:])
    variant = p.parts[-2]
    completed.add((dataset, H, variant))

missing = [
    (d, H, v)
    for d in datasets
    for H in horizons
    for v in variants
    if (d, H, v) not in completed
]

print("\nMissing runs:", len(missing))
for x in missing:
    print(x)

print("\nLatest saved results:")
for p in sorted(files, key=lambda x: x.stat().st_mtime)[-10:]:
    t = datetime.fromtimestamp(p.stat().st_mtime)
    print(t.strftime("%Y-%m-%d %H:%M:%S"), p)

aggregate = BASE / "strong_sparse_mask_screening_results.csv"
failures = BASE / "failures.csv"

print("\nFinal aggregate exists:", aggregate.exists())
print("Failures file exists:", failures.exists())

if aggregate.exists():
    df = pd.read_csv(aggregate)
    print("Rows in aggregate:", len(df))
    display(df.tail())

Completed result files: 60/60

Missing runs: 0

Latest saved results:
2026-08-22 03:35:24 /data/code/2026_08/results_itransformer_explicit_sparse_horizon_mask_all20/ETTm1/H96/HorizonAdaptiveSparse/result.csv
2026-08-22 03:35:51 /data/code/2026_08/results_itransformer_explicit_sparse_horizon_mask_all20/ETTm1/H192/DenseFineTune/result.csv
2026-08-22 03:36:33 /data/code/2026_08/results_itransformer_explicit_sparse_horizon_mask_all20/ETTm1/H192/SharedSparse/result.csv
2026-08-22 03:37:14 /data/code/2026_08/results_itransformer_explicit_sparse_horizon_mask_all20/ETTm1/H192/HorizonAdaptiveSparse/result.csv
2026-08-22 03:37:41 /data/code/2026_08/results_itransformer_explicit_sparse_horizon_mask_all20/ETTm1/H336/DenseFineTune/result.csv
2026-08-22 03:38:30 /data/code/2026_08/results_itransformer_explicit_sparse_horizon_mask_all20/ETTm1/H336/SharedSparse/result.csv
2026-08-22 03:39:04 /data/code/2026_08/results_itransformer_explicit_sparse_horizon_mask_all20/ETTm1/H336/HorizonAdaptiveSparse/res

,dataset,pred_len,variant,C,K,shared_adaptive_jaccard,batch_size,new_trainable_params,checkpoint_fidelity_abs_diff,base_val_full_mse,...,best_val_full_mse,best_epoch,test_full_mse,test_full_mae,test_endpoint_mse,test_endpoint_mae,full_mse_gain_vs_base_%,endpoint_mse_gain_vs_base_%,val_improves_base,elapsed_sec
55,ETTm1,336,SharedSparse,7,3,0.785714,32,1750352,0.0,0.665517,...,0.668790,3,0.470353,0.434133,0.592232,0.492095,-9.921049,-13.278243,0,48.772099
56,ETTm1,336,HorizonAdaptiveSparse,7,3,0.785714,32,1750352,0.0,0.665517,...,0.667390,1,0.454671,0.430514,0.571921,0.485735,-6.256195,-9.393298,0,33.255906
57,ETTm1,720,DenseFineTune,7,3,0.857143,32,1947344,0.0,0.973418,...,0.973418,0,0.503875,0.463796,0.624105,0.526396,-0.000184,0.000022,0,23.367231
58,ETTm1,720,SharedSparse,7,3,0.857143,32,1947344,0.0,0.973418,...,0.973119,3,0.516340,0.464434,0.633886,0.521747,-2.474005,-1.567252,1,46.086331
59,ETTm1,720,HorizonAdaptiveSparse,7,3,0.857143,32,1947344,0.0,0.973418,...,0.970546,3,0.515778,0.465143,0.640278,0.525046,-2.362535,-2.591346,1,45.860007


# 21. How to interpret the three possible outcomes

This section implements the matched horizon-specific topology-transfer mechanism. Predictive-utility source sets are computed from training data and then imposed on the neural forecaster as specified in the paper.
